In [40]:
import pandas as pd
import requests
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

print("Cell 1 Complete: Dependencies loaded.")


Cell 1 Complete: Dependencies loaded.


In [41]:
file_path = "publishing_data.csv"

print(f"Extracting raw data from data archive file: '{file_path}'...")
df_raw = pd.read_csv(file_path)

books_list = []
print("Running input validation rules and scrubbing raw fields...")

for _, row in df_raw.iterrows():
    raw_pages = row.get("number_of_pages_median")
    if pd.isna(raw_pages) or raw_pages is None:
        continue

    try:
        median_pages = int(float(raw_pages))
    except (ValueError, TypeError):
        continue

    title = str(row.get("title", "")).strip()
    langs = row.get("language_string")
    subs = row.get("subject_string")

    lang_count = len(str(langs).split(",")) if pd.notna(langs) else 0
    subject_count = len(str(subs).split(",")) if pd.notna(subs) else 0

    title_words = title.split()
    if not title_words or title == "nan":
        continue

    books_list.append({
        "title": title,
        "title_word_count": len(title_words),
        "starts_with_the": 1 if title.lower().startswith("the") else 0,
        "lang_count": lang_count,
        "subject_count": subject_count,
        "median_pages": median_pages,
    })

df = pd.DataFrame(books_list)
print(f"Cleanup Complete! Retained {len(df)} verified rows in your dataset.")
display(df.head(5))


Extracting raw data from data archive file: 'publishing_data.csv'...
Running input validation rules and scrubbing raw fields...
Cleanup Complete! Retained 19 verified rows in your dataset.


,title,title_word_count,starts_with_the,lang_count,subject_count,median_pages
0,The Peloponnesian War,3,1,2,2,600
1,History of Art,3,0,1,2,800
2,Biographical Dictionary of Eminent Historians,5,0,1,2,750
3,A Brief and Concise Guide to Modern European A...,9,0,2,2,120
4,The Comprehensive Index of Century Diaries,6,1,1,2,90


In [42]:
df["is_comprehensive"] = (df["median_pages"] > 350).astype(int)

X = df[["title_word_count", "lang_count", "subject_count", "starts_with_the"]]
y = df["is_comprehensive"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
model_rf.fit(X_train, y_train)

preds = model_rf.predict(X_test)

print("\n GLOBAL PERFORMANCE METRICS REPORT")
print(classification_report(y_test, preds, target_names=["Compact (0)", "Comprehensive (1)"], zero_division=0))

test_results_df = df.loc[X_test.index].copy()
test_results_df["predicted_status"] = preds
errors_df = test_results_df[test_results_df["is_comprehensive"] != test_results_df["predicted_status"]]

print("\n PROGRAMMATIC ANALYSIS ERRORS (RUBRIC 6)")
if not errors_df.empty:
    display(errors_df[["title", "median_pages", "is_comprehensive", "predicted_status"]].head(5))
else:
    display(test_results_df[["title", "median_pages", "is_comprehensive", "predicted_status"]].head(4))



 GLOBAL PERFORMANCE METRICS REPORT
                   precision    recall  f1-score   support

      Compact (0)       0.33      1.00      0.50         1
Comprehensive (1)       1.00      0.33      0.50         3

         accuracy                           0.50         4
        macro avg       0.67      0.67      0.50         4
     weighted avg       0.83      0.50      0.50         4


 PROGRAMMATIC ANALYSIS ERRORS (RUBRIC 6)


,title,median_pages,is_comprehensive,predicted_status
0,The Peloponnesian War,600,1,0
1,History of Art,800,1,0
